In [2]:
import re
import collections

def get_word_frequencies(sentences):
    """
    Preprocesses sentences and returns a frequency dictionary of words.
    Lowercase, remove periods, and add </w> to each word.
    """
    word_freq = collections.defaultdict(int)
    for sentence in sentences:
        # Lowercase, remove period, and split into words
        words = sentence.lower().replace('.', '').split()
        for word in words:
            # Append end-of-word token and count frequency
            word_freq[word] += 1
    return word_freq

def initialize_corpus(word_freq):
    """
    Splits each word into characters, creating the initial corpus representation.
    Example: {'the</w>': 8} -> {'t h e </w>': 8}
    """
    corpus = {}
    for word, freq in word_freq.items():
        corpus[" ".join(list(word))] = freq
    return corpus

def initialize_vocab(corpus):
    """Initializes the vocabulary with all unique characters."""
    vocab = set()
    for word in corpus:
        vocab.update(word.split())
    return list(vocab)

def get_pair_stats(corpus):
    """Counts the frequency of each adjacent pair of symbols."""
    pairs = collections.defaultdict(int)
    for word, freq in corpus.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            # Add the frequency of the word to the pair's count
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_pair(best_pair, corpus):
    """Merges the most frequent pair in the corpus."""
    new_corpus = {}
    # Create the merged token (e.g., ('e', '</w>') -> 'e</w>')
    new_token = "".join(best_pair)
    # Use regex to replace the pair in the keys of the corpus dictionary
    # We escape special characters and handle the space between tokens
    pattern = re.compile(r'(?<!\S)' + re.escape(' '.join(best_pair)) + r'(?!\S)')
    
    for word, freq in corpus.items():
        new_word = pattern.sub(new_token, word)
        new_corpus[new_word] = freq
        
    return new_corpus

def tokenize_word(input_word, vocab):
    """
    Tokenizes a word using the greedy longest-match algorithm.
    """
    # Ensure the word has the end-of-word token
    # if not input_word.endswith('</w>'):
    #     input_word += '</w>'

    tokens = []
    current_pos = 0
    while current_pos < len(input_word):
        # Find the longest subword in the vocab that is a prefix
        best_match = ""
        for i in range(current_pos, len(input_word)):
            subword = input_word[current_pos : i+1]
            if subword in vocab:
                best_match = subword
        
        # If no match is found (should only happen for unknown chars),
        # treat the single character as the token.
        if not best_match:
            best_match = input_word[current_pos]

        tokens.append(best_match)
        current_pos += len(best_match)
        
    return tokens

# --- Main execution ---
if __name__ == "__main__":
    # 1. Dataset Initialization
    dataset = [
        "The boy hugs the cat.",
        "The boys are hugging the dogs.",
        "The dogs are chasing the cats.",
        "The dog and the cat sit quietly.",
        "The boy is sitting on the dog."
    ]
    num_merges = 20

    # 2. Preprocessing
    word_frequencies = get_word_frequencies(dataset)
    corpus = initialize_corpus(word_frequencies)
    vocab = initialize_vocab(corpus)

    print("--- Initial State ---")
    print(f"Initial Vocabulary Size: {len(vocab)}")
    # print(f"Initial Vocabulary: {sorted(vocab)}")
    # print(f"Initial Corpus: {corpus}")
    print("-" * 20)

    # 3. Training Loop (Applying Merges)
    for i in range(num_merges):
        pair_stats = get_pair_stats(corpus)
        if not pair_stats:
            break # No more pairs to merge
        
        # Find the most frequent pair
        # The key for max is a lambda function to look up the frequency in pair_stats
        # Tie-breaking is handled implicitly by Python's default behavior for max()
        best_pair = max(pair_stats, key=pair_stats.get)
        
        # Perform the merge
        corpus = merge_pair(best_pair, corpus)
        
        # Add the new merged token to the vocabulary
        new_token = "".join(best_pair)
        vocab.append(new_token)
        
        print(f"Iteration {i+1}: Merged {best_pair} -> '{new_token}' (Frequency: {pair_stats[best_pair]})")

    # 4. Final Vocabulary
    print("\n--- Final Vocabulary ---")
    print(f"Final Vocabulary Size: {len(vocab)}")
    print(sorted(vocab))
    print("-" * 20)

    # 5. Tokenize a new sentence
    new_sentence = "The cat is chasing the dog quietly."
    print(f"\n--- Tokenizing New Sentence ---\n'{new_sentence}'\n")

    tokenized_sentence = []
    for word in new_sentence.lower().replace('.', '').split():
        tokens = tokenize_word(word, vocab)
        tokenized_sentence.append(tokens)
        print(f"'{word}' -> {tokens}")
    
    print("\nFinal Tokenized Output:", tokenized_sentence)


--- Initial State ---
Initial Vocabulary Size: 17
--------------------
Iteration 1: Merged ('t', 'h') -> 'th' (Frequency: 10)
Iteration 2: Merged ('th', 'e') -> 'the' (Frequency: 10)
Iteration 3: Merged ('d', 'o') -> 'do' (Frequency: 4)
Iteration 4: Merged ('do', 'g') -> 'dog' (Frequency: 4)
Iteration 5: Merged ('b', 'o') -> 'bo' (Frequency: 3)
Iteration 6: Merged ('bo', 'y') -> 'boy' (Frequency: 3)
Iteration 7: Merged ('c', 'a') -> 'ca' (Frequency: 3)
Iteration 8: Merged ('ca', 't') -> 'cat' (Frequency: 3)
Iteration 9: Merged ('i', 'n') -> 'in' (Frequency: 3)
Iteration 10: Merged ('in', 'g') -> 'ing' (Frequency: 3)
Iteration 11: Merged ('h', 'u') -> 'hu' (Frequency: 2)
Iteration 12: Merged ('hu', 'g') -> 'hug' (Frequency: 2)
Iteration 13: Merged ('a', 'r') -> 'ar' (Frequency: 2)
Iteration 14: Merged ('ar', 'e') -> 'are' (Frequency: 2)
Iteration 15: Merged ('dog', 's') -> 'dogs' (Frequency: 2)
Iteration 16: Merged ('s', 'i') -> 'si' (Frequency: 2)
Iteration 17: Merged ('si', 't') -> 's